<a href="https://colab.research.google.com/github/Youssif-Kady/flayrank_task1/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [ ]:
import pandas as pd
import numpy as np
import os
import glob

# 1. Download and load the data since this is a fresh Colab session
if not os.path.exists('flyrank-ml-internship-starter'):
    !git clone https://github.com/flyrank-bih/flyrank-ml-internship-starter.git

data_files = glob.glob('flyrank-ml-internship-starter/**/*.parquet', recursive=True) + \
             glob.glob('flyrank-ml-internship-starter/**/*.csv', recursive=True)

if data_files:
    file_path = data_files[0]
    df = pd.read_parquet(file_path) if file_path.endswith('.parquet') else pd.read_csv(file_path)
    print("✅ Data loaded successfully!\n")
else:
    raise FileNotFoundError("Dataset not found. Check the repo clone.")

# 2. Build the feature vector
def build_features(data):
    df_features = data.copy()

    # Select ONLY raw, safe input features.
    safe_features = [
        'impressions_90d', 'clicks_90d', 'sessions_90d', 'avg_position', 'ctr',
        'content_age_days', 'days_since_last_update', 'word_count',
        'trend_direction', 'competition_level', 'content_type', 'main_intent'
    ]
    target = 'is_declining_label'

    # Keep only columns that exist
    cols_to_keep = [col for col in safe_features + [target] if col in df_features.columns]
    df_features = df_features[cols_to_keep]

    # Fill missing values with 0
    num_cols = df_features.select_dtypes(include=[np.number]).columns
    df_features[num_cols] = df_features[num_cols].fillna(0)

    # Categorical Handling (One-Hot Encoding)
    cat_cols = ['trend_direction', 'competition_level', 'content_type', 'main_intent']
    cat_cols_present = [c for c in cat_cols if c in df_features.columns]

    if cat_cols_present:
        df_features = pd.get_dummies(df_features, columns=cat_cols_present, drop_first=True)

    return df_features

# Build and preview the vector
df_vector = build_features(df)
print(f"Feature vector shape: {df_vector.shape}")
df_vector.head(3)

✅ Data loaded successfully!

Feature vector shape: (30000, 19)


,impressions_90d,clicks_90d,sessions_90d,avg_position,ctr,content_age_days,days_since_last_update,word_count,trend_direction_flat,trend_direction_new,trend_direction_stable,trend_direction_up,competition_level_LOW,competition_level_MEDIUM,content_type_feedly article,content_type_keyword article,main_intent_informational,main_intent_navigational,main_intent_transactional
0,3803,29,17,10.6,0.76,187,20,3221.0,False,False,False,False,False,False,False,True,False,False,True
1,15320,7,9,20.3,0.05,445,25,2481.0,False,False,False,False,True,False,False,True,True,False,False
2,12581,11,11,36.5,0.09,141,20,3515.0,False,False,False,False,True,False,False,True,True,False,False


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

impressions_90d, clicks_90d, sessions_90d, avg_position, ctr:

Meaning: Raw performance metrics over the last 90 days.

Missing: Handled by filling with 0 to represent lack of traffic/data.

Available-when: Yes, fully available from analytics tools on the day before the prediction.

content_age_days, days_since_last_update, word_count:

Meaning: Content structure and temporal properties.

Missing: Filled with 0.

Available-when: Always available directly from the CMS metadata.

Categoricals (e.g., trend_direction_*, competition_level_*):

Meaning: SEO context and intent classification, one-hot encoded for model compatibility.

Missing: Handled naturally by dummy encoding (creates 0s across all active flags if missing).

Available-when: Static or pre-calculated historical flags, safely available before prediction.

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [ ]:
# 1. Create our proxy target label 'is_declining' based on the trend direction
# We classify the page as declining (1) if the trend_direction is 'down', else (0)
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)
target_col = 'is_declining'

print(f"Target column mathematically created: '{target_col}'")

# 2. Temporarily add the target to our feature vector to calculate correlation
df_corr = df_vector.copy()
df_corr[target_col] = df[target_col]

# 3. Calculate correlations
correlations = df_corr.corr(numeric_only=True)[target_col].sort_values(ascending=False)

print("\nTop Positive Correlations with Target:")
print(correlations.head(5))
print("\nTop Negative Correlations with Target:")
print(correlations.tail(5))

# 4. Flag anything highly correlated (>0.75 or <-0.75) as a potential leak
leakage_suspects = correlations[(correlations > 0.75) | (correlations < -0.75)].index.tolist()
if target_col in leakage_suspects:
    leakage_suspects.remove(target_col)

if len(leakage_suspects) > 0:
    print(f"\n⚠️ WARNING: Potential leakage detected in features: {leakage_suspects}. Investigate!")
else:
    print("\n✅ No obvious leakage found via correlation. Selected features look clean.")

Target column mathematically created: 'is_declining'

Top Positive Correlations with Target:
is_declining                    1.000000
word_count                      0.118863
content_type_keyword article    0.118346
days_since_last_update          0.081383
competition_level_LOW           0.076146
Name: is_declining, dtype: float64

Top Negative Correlations with Target:
content_age_days         -0.163882
trend_direction_flat     -0.217417
trend_direction_new      -0.308759
trend_direction_up       -0.450336
trend_direction_stable   -0.541841
Name: is_declining, dtype: float64

✅ No obvious leakage found via correlation. Selected features look clean.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

trend_direction: Excluded because it was directly used to derive our proxy target label is_declining. Keeping its dummy variables (like trend_direction_up or stable) would mathematically leak the target to the model.

Identifiers (content_id, client_id): Excluded because the model should learn generalized performance patterns, not memorize specific URLs or clients.

Simultaneous Performance Metrics (Current day impressions/clicks): Excluded to strictly ensure we only predict using historical (T-1) data, preventing future data leakage

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.